# Overview

This notebook runs the setup for the agent. It guides you through the following steps:

1. **Schema and Volume Setup:** Creates a dedicated schema and volume for your data.
2. **Permissions:** Grants necessary permissions to all account users for collaborative access.
3. **Data Upload:** Uploads sample CSV files to the Databricks volume.
4. **Delta Table Creation:** Reads the uploaded CSV files and creates Delta tables, including enabling Change Data Feed for the `product_docs` table.
5. **Vector Search Endpoint:** Initializes and waits for a Databricks Vector Search endpoint.
6. **Index Creation:** Builds a vector search index on the `product_docs` table using the `databricks-gte-large-en` embedding model.

By the end of this workflow, all the data dependencies for the agent are setup

In [0]:
%pip install -U -qqqq unitycatalog-ai[databricks] mlflow-skinny[databricks] langgraph==0.3.4 databricks-langchain databricks-agents uv
%restart_python

In [0]:
import sys
sys.path.append(".")

from config import *

In [0]:
%sql
create schema if not exists identifier(:WORKSHOP_CATALOG || "." || :WORKSHOP_SCHEMA);
create volume if not exists identifier(:WORKSHOP_CATALOG || "." || :WORKSHOP_SCHEMA || ".data");
use identifier(:WORKSHOP_CATALOG || "." || :WORKSHOP_SCHEMA);

In [0]:
spark.sql(
    f"grant use schema, select, execute, read volume on schema {WORKSHOP_CATALOG}.{WORKSHOP_SCHEMA} to `account users`"
)

In [0]:
import os

WORKSPACE_DIR = os.getcwd()
DATA_PATH = f"{WORKSPACE_DIR}/data"
VOLUME_PATH = f"/Volumes/{os.environ['WORKSHOP_CATALOG']}/{os.environ['WORKSHOP_SCHEMA']}/data"

print(f"Removing existing CSVs from {VOLUME_PATH}...")
os.system(f"rm -f {VOLUME_PATH}/*.csv")

print(f"Copying CSVs from {DATA_PATH} to {VOLUME_PATH}...")
os.system(f"cp {DATA_PATH}/*.csv {VOLUME_PATH}")

In [0]:
%sql
-- demo_shipments.csv -> table: shipments
CREATE OR REPLACE TABLE shipments
SELECT * EXCEPT (_rescued_data)
FROM read_files(
  "/Volumes/" || :WORKSHOP_CATALOG || "/" || :WORKSHOP_SCHEMA || "/data/demo_shipments.csv",
  FORMAT => "csv",
  HEADER => true
);

-- demo_suppliers.csv -> table: suppliers
CREATE OR REPLACE TABLE suppliers
SELECT * EXCEPT (_rescued_data)
FROM read_files(
  "/Volumes/" || :WORKSHOP_CATALOG || "/" || :WORKSHOP_SCHEMA || "/data/demo_suppliers.csv",
  FORMAT => "csv",
  HEADER => true
);

-- demo_inventory.csv -> table: inventory
CREATE OR REPLACE TABLE inventory
SELECT * EXCEPT (_rescued_data)
FROM read_files(
  "/Volumes/" || :WORKSHOP_CATALOG || "/" || :WORKSHOP_SCHEMA || "/data/demo_inventory.csv",
  FORMAT => "csv",
  HEADER => true
);

-- demo_supplier_sops.csv -> table: supplier_sops  (multiline text with embedded quotes)
CREATE OR REPLACE TABLE supplier_sops
TBLPROPERTIES (delta.enableChangeDataFeed = true)
SELECT * EXCEPT (_rescued_data)
FROM read_files(
  "/Volumes/" || :WORKSHOP_CATALOG || "/" || :WORKSHOP_SCHEMA || "/data/demo_supplier_sops.csv",
  FORMAT => "csv",
  HEADER => true,
  MULTILINE => true,
  ESCAPE => '"'
);

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

if WORKSHOP_CATALOG not in [v["name"] for v in client.list_endpoints().get("endpoints", [])]:
    print(f"Creating endpoint {WORKSHOP_CATALOG}")
    client.create_endpoint(name=WORKSHOP_CATALOG, endpoint_type="STANDARD")

client.wait_for_endpoint(WORKSHOP_CATALOG)
print(f"Created endpoint {WORKSHOP_CATALOG}")

In [0]:
# index_name = f"{WORKSHOP_CATALOG}.{WORKSHOP_SCHEMA}.product_docs_index"
# if index_name not in [
#     i["name"] for i in client.list_indexes(WORKSHOP_CATALOG).get("vector_indexes", [])
# ]:
#   print(f"Creating index {index_name}")
#   client.create_delta_sync_index_and_wait(
#         endpoint_name=WORKSHOP_CATALOG,
#         source_table_name=f"{WORKSHOP_CATALOG}.{WORKSHOP_SCHEMA}.product_docs",
#         index_name=index_name,
#         pipeline_type="TRIGGERED",
#         primary_key="product_id",
#         embedding_source_column="indexed_doc",
#         embedding_model_endpoint_name="databricks-gte-large-en",
#     )
# print(f"Created index {index_name}")